In [31]:
import pandas as pd
df=pd.read_csv('IMDB Dataset.csv')
df.head() #ilk 5 satir

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [32]:
#metinler küçük harfe çevirmeliyiz
df['review']=df['review'].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [33]:
# 'review' sütunundaki '<br />' ifadelerini bul ve içini boşluk (' ') ile değiştiriyoruz
# regex=False diyerek bunun özel bir kod değil, düz metin olduğunu belirtiyoruz.
df['review'] = df['review'].str.replace('<br />', ' ', regex=False)
# 1. indexli yoruma (az önce <br /> olan yoruma) tekrar bakalım
df['review'][1]


'a wonderful little production.   the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece.   the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life.   the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well done.'

In [34]:
# 'review' sütunundaki harf ve sayı dışındaki HER ŞEYİ (noktalama işaretlerini) bulup,
# içlerini hiçbir şey ('') ile değiştirip (yani silip) aynı sütuna yazıyoruz.
# [^\w\s] kodu: "Harf (w) ve Boşluk (s) OLMAYAN (^) her şeyi bul" anlamına gelir.

df['review'] = df['review'].str.replace(r'[^\w\s]', '', regex=True)

# İşe yaradı mı diye görmek için o 1. indexli yoruma tekrar bakıyoruz
df['review'][1]

'a wonderful little production   the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece   the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices down pat too you can truly see the seamless editing guided by the references to williams diary entries not only is it well worth the watching but it is a terrificly written and performed piece a masterful production about one of the great masters of comedy and his life   the realism really comes home with the little things the fantasy of the guard which rather than use the traditional dream techniques remains solid then disappears it plays on our knowledge and our senses particularly with the scenes concerning orton and halliwell and the sets particularly of their flat with halliwells murals decorating every surface are terribly well done'

In [35]:
#eğitim ve testi bölme aşaması
from sklearn.model_selection import train_test_split

# 1. x ve y yi belirliyoruz
X=df['review']
y=df['sentiment']

# 2. Veriyi %80 Eğitim (Train), %20 Test olarak ayırıyoruz.
# random_state=42: Verileri rastgele karıştırırken ekranda aynı soruların seçilmesi için bir kilit numarasıdır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Kaç soruyu makineye vereceğiz, kaç soruyu saklayacağız
print("Eğitim için ayrılan yorum sayısı:", len(X_train))
print("Sınav (Test) için saklanan yorum sayısı:", len(X_test))

Eğitim için ayrılan yorum sayısı: 40000
Sınav (Test) için saklanan yorum sayısı: 10000


In [36]:

from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer(max_features=5000)
#EĞİTİM (TRAIN) VERİSİNİ DÖNÜŞTÜRME
# fit: Sadece eğitim verisine bakarak o en popüler 5000 kelimelik sözlüğü çıkar (öğren).
# transform: Öğrendiği sözlüğe göre metinleri 0 ve 1'lerden oluşan sayılara dönüştür.
# fit_transform ikisini aynı anda yapar. Makinenin ders çalıştığı yer burasıdır.
X_train_sayisal=vectorizer.fit_transform(X_train)


In [37]:
#TEST (SINAV) VERİSİNİ DÖNÜŞTÜRME
#Sadece bildiklerini kullanarak metni sayılara çevirmesini söylüyoruz.
X_test_sayisal=vectorizer.transform(X_test)

In [38]:
#KONTROL AŞAMASI
print("Eğitim tablomuzun boyutu:", X_train_sayisal.shape)

Eğitim tablomuzun boyutu: (40000, 5000)


In [39]:
# Kullanacağımız algoritmayı ve başarı ölçüm aletini çağırıyoruz
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
#Makineye matematiği çözmesi için 1000 defaya kadar deneme izni veriyoruz.
model=LogisticRegression(max_iter=1000)


In [40]:
# .fit komutuyla makineye 40.000 soruyu (X_train_sayisal) ve (y_train) veriyoruz.
print("Model eğitiliyor, lütfen 10-15 saniye bekle...")
model.fit(X_train_sayisal, y_train)

Model eğitiliyor, lütfen 10-15 saniye bekle...


LogisticRegression(max_iter=1000)

In [41]:
# Makineye daha önce hiç görmediği 10.000 sınav sorusunu (X_test_sayisal) veriyoruz.
# .predict komutuyla bu soruların cevaplarını tahmin etmesini istiyoruz.
tahminler = model.predict(X_test_sayisal)

In [42]:
#makinenin tahminleri ile bizdeki gizli cevap anahtarını (y_test) karşılaştırıyoruz.
basari_orani = accuracy_score(y_test, tahminler)

print("Modelin Sınav Başarı Oranı: %", basari_orani * 100)

Modelin Sınav Başarı Oranı: % 88.3


In [43]:
# 1. Kendi test yorumlarımı yazıyorum
benim_yorumlarım = [
    "I really loved this movie, the acting was amazing and beautiful", # Olumlu cümlemiz
    "What a waste of time, terrible plot and bad actors"               # Olumsuz cümlemiz
]


In [44]:
# 2. Metinlerimizi sayılara çeviriyoruz.
yorumlar_sayisal = vectorizer.transform(benim_yorumlarım)
tahminler = model.predict(yorumlar_sayisal)
tahminler = model.predict(yorumlar_sayisal)

print("1. Yorumun Duygusu Tahmini:", tahminler[0])
print("2. Yorumun Duygusu Tahmini:", tahminler[1])

1. Yorumun Duygusu Tahmini: positive
2. Yorumun Duygusu Tahmini: negative


In [45]:
import gradio as gr

# 1. Tahmin Fonksiyonumuz (İkonları da mavi/turuncu konseptine uydurduk)
def duygu_tahmini(kullanici_yorumu):
    if not kullanici_yorumu.strip():
        return "⚠️ Lütfen analizi yapılacak bir metin girin."

    yorum_sayisal = vectorizer.transform([kullanici_yorumu])
    tahmin = model.predict(yorum_sayisal)[0]

    if tahmin == 'positive':
        return "🔵 Olumlu (Positive) - İzlemeye Değer!"
    else:
        return "🟠 Olumsuz (Negative) - Zaman Kaybı Olabilir!"

# 2. Mavi - Turuncu Tema Ayarları
# primary_hue="orange" diyerek ana butonları turuncu yapıyoruz.
# neutral_hue="blue" diyerek arka plan/kenarlık tonlarına hafif mavilik katıyoruz.
tema = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="blue",
    neutral_hue="slate"
)

with gr.Blocks(theme=tema) as arayuz:

    # Başlık ve açıklamayı Markdown yerine HTML ile yazıp renklendiriyoruz
    # #1E3A8A (Koyu Mavi) ve #EA580C (Canlı Turuncu)
    gr.HTML("""
        <div style='text-align: center; margin-bottom: 20px;'>
            <h1 style='color: #1E3A8A; font-family: sans-serif;'>🎬 Film Yorumu Duygu Analizi</h1>
            <p style='color: #EA580C; font-size: 16px; font-weight: bold;'>
                50.000 IMDB film yorumu kullanılarak eğitilmiş Makine Öğrenmesi Modeli
            </p>
        </div>
    """)

    with gr.Row():

        with gr.Column():
            yorum_girisi = gr.Textbox(
                label="İngilizce Yorumunuzu Buraya Yazın",
                lines=5,
                placeholder="Örnek: This movie is an absolute masterpiece! The acting and directing are flawless..."
            )
            with gr.Row():
                temizle_butonu = gr.ClearButton(components=[yorum_girisi], value="🗑️ Temizle")
                # primary varyantı sayesinde bu buton belirlediğimiz 'orange' rengini alacak
                tahmin_butonu = gr.Button("🔍 Analiz Et", variant="primary")

        with gr.Column():
            sonuc_ciktisi = gr.Textbox(
                label="Yapay Zeka Kararı",
                lines=3,
                interactive=False
            )

    tahmin_butonu.click(fn=duygu_tahmini, inputs=yorum_girisi, outputs=sonuc_ciktisi)

# 3. Arayüzü Başlat!
arayuz.launch(share=True)

/tmp/ipykernel_747/441618245.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=tema) as arayuz:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a1e7e30a673f857071.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
